# 10_graph_cypher_qa

10_graph_cypher_qa.py — ★ 메인: 자연어 질문 → Cypher → 그래프 답변

GraphCypherQAChain 은 (1) LLM 이 스키마 보고 Cypher 생성 → (2) Neo4j 실행 → (3) LLM 이 결과를 자연어로 정리.

이 스크립트를 실행하기 전에 `python 08_load_to_neo4j.py` 가 먼저 돌아 있어야 그래프가 채워져 있다.

In [1]:
import os, sys, ssl, certifi
# Windows 인증서 저장소 손상 우회(임베딩/HTTPS 로드 SSL 에러 방지)
ssl.SSLContext.load_default_certs = lambda self, *a, **k: self.load_verify_locations(certifi.where())
# 노트북 커널엔 __file__ 이 없으므로 스크립트 호환 위해 정의 + supp/ 를 import 경로에 추가
__file__ = os.path.join(os.getcwd(), '10_graph_cypher_qa.py')
sys.path.insert(0, os.path.abspath('..'))

In [2]:
"""
10_graph_cypher_qa.py — ★ 메인: 자연어 질문 → Cypher → 그래프 답변

GraphCypherQAChain 은 (1) LLM 이 스키마 보고 Cypher 생성 → (2) Neo4j 실행 → (3) LLM 이 결과를 자연어로 정리.

이 스크립트를 실행하기 전에 `python 08_load_to_neo4j.py` 가 먼저 돌아 있어야 그래프가 채워져 있다.
"""
import sys as _sys
from pathlib import Path as _Path
_sys.path.insert(0, str(_Path(__file__).resolve().parent.parent))

from langchain_neo4j import GraphCypherQAChain

from _common import get_llm, get_neo4j_graph, banner, llm_unavailable


QUESTIONS = [
    "Anthropic을 설립한 사람은 누구인가?",
    "Anthropic이 개발한 제품은?",
    "Anthropic 에 투자한 회사는?",
]


def main() -> None:
    banner("GraphCypherQAChain — 자연어 → Cypher → 답변")
    llm = get_llm()
    if llm is None:
        llm_unavailable()
        return

    graph = get_neo4j_graph()  # refresh_schema=True
    if graph is None:
        print("❌ NEO4J_* env 미설정")
        return

    chain = GraphCypherQAChain.from_llm(
        llm=llm,
        graph=graph,
        verbose=True,
        allow_dangerous_requests=True,  # LLM 생성 Cypher 의 임의 실행을 명시적으로 허용
    )

    for q in QUESTIONS:
        print("\n" + "─" * 70)
        print(f"❓ {q}")
        try:
            result = chain.invoke({"query": q})
            print(f"💬 {result['result']}")
        except Exception as e:
            print(f"⚠ {type(e).__name__}: {str(e)[:200]}")


if __name__ == "__main__":
    main()

D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



📌 GraphCypherQAChain — 자연어 → Cypher → 답변



──────────────────────────────────────────────────────────────────────
❓ Anthropic을 설립한 사람은 누구인가?


> Entering new GraphCypherQAChain chain...


Generated Cypher:
MATCH (p:Person)-[:FOUNDED]->(c:Company {id: 'Anthropic'})
RETURN p.id
Full Context:
[{'p.id': 'Dario Amodei'}, {'p.id': 'Daniela Amodei'}]



> Finished chain.
💬 Anthropic을 설립한 사람은 Dario Amodei와 Daniela Amodei입니다.

──────────────────────────────────────────────────────────────────────
❓ Anthropic이 개발한 제품은?


> Entering new GraphCypherQAChain chain...


Generated Cypher:
MATCH (c:Company {id: 'Anthropic'})-[:DEVELOPED]->(p:Product)
RETURN p.id
Full Context:
[{'p.id': 'Claude'}]



> Finished chain.
💬 Anthropic이 개발한 제품은 Claude입니다.

──────────────────────────────────────────────────────────────────────
❓ Anthropic 에 투자한 회사는?


> Entering new GraphCypherQAChain chain...


Generated Cypher:
MATCH (investor:Company)-[:INVESTED_IN]->(target:Company {id: 'Anthropic'})
RETURN investor.id
Full Context:
[{'investor.id': 'Amazon'}]



> Finished chain.
💬 Anthropic에 투자한 회사는 Amazon입니다.
